# 02 — Middleware: the customization escape hatch

Middleware is LangChain 1.0's answer to the old "controllability wall" —
you can now hook into the agent loop (before/after the model call,
around tool execution, before/after the whole run) instead of
rewriting the whole abstraction.

This notebook covers:
- Built-in middleware (`SummarizationMiddleware`, `PIIMiddleware`, call limits)
- Writing your own custom middleware
- Stacking multiple middlewares on one agent


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


In [2]:
@tool
def lookup_account(account_id: str) -> str:
    """Look up a fake account record by id."""
    return f"Account {account_id}: balance=₹42,500, status=active, email=user{account_id}@example.com"


## 1. Built-in middleware

- `SummarizationMiddleware` — compresses old messages so long threads don't overflow context
- `PIIMiddleware` — redacts sensitive fields (emails, phone numbers, etc.) before they reach the model or the log
- `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` — cost / runaway-loop guards

> Import paths can shift between minor releases — if an import below
> fails, check `from langchain.agents.middleware import ...` against
> your installed version with `pip show langchain`.


In [3]:
# ============ BUILT-IN MIDDLEWARE ============
from langchain.agents.middleware import (
    SummarizationMiddleware,
    PIIMiddleware,
    ModelCallLimitMiddleware,
)

agent_with_builtins = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[
        # One PIIMiddleware per PII type - the type is the first positional
        # arg, not a `pii_types` list. Built-ins: email, credit_card, ip,
        # mac_address, url (or pass your own regex/callable via `detector`).
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Needs its own model to write the summary with, and the threshold
        # goes in `trigger` - {"tokens": n} / {"messages": n} / {"fraction": f}.
        SummarizationMiddleware(
            model=llm,
            trigger={"tokens": 2000},
            keep=("messages", 20),
        ),
        # Caps model calls: `run_limit` per invocation, `thread_limit`
        # across a whole conversation thread.
        ModelCallLimitMiddleware(run_limit=5),
    ],
)

# Do NOT end the cell with a bare `agent_with_builtins`. Jupyter would call the
# object's `_repr_mimebundle_`, which calls `draw_mermaid_png()` -> POSTs the
# diagram to the mermaid.ink web service -> ValueError when it's unreachable.
# Printing the graph structure is local and needs no network:
print(type(agent_with_builtins).__name__)
print("Nodes:", list(agent_with_builtins.get_graph().nodes))

# Want the diagram? `.draw_mermaid()` returns the source text locally; only the
# `_png` variants call out to the network.
# print(agent_with_builtins.get_graph().draw_mermaid())

CompiledStateGraph
Nodes: ['__start__', 'model', 'tools', 'PIIMiddleware[email].before_model', 'PIIMiddleware[email].after_model', 'SummarizationMiddleware.before_model', 'ModelCallLimitMiddleware.before_model', 'ModelCallLimitMiddleware.after_model', '__end__']


In [4]:
agent_with_builtins

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [5]:
result = agent_with_builtins.invoke({
    "messages": [{"role": "user", "content": "Look up account 1029 and tell me the balance."}]
})

for m in result["messages"]:
    print(f"[{m.type}] {m.content}")


[human] Look up account 1029 and tell me the balance.
[ai] 
[tool] Account 1029: balance=₹42,500, status=active, email=user1029@example.com
[ai] The balance for account 1029 is ₹42,500. Is there anything else you would like to know?


## 2. Writing your own middleware

Custom middleware subclasses `AgentMiddleware` and overrides the hook for the
point in the loop you want to intercept. Two rules matter and are easy to get
wrong:

1. **Every hook takes `(self, state, runtime)`.** The `runtime` argument is not
   optional — a hook defined as `before_model(self, state)` raises `TypeError`
   at run time.
2. **Hooks return a state *update*, they do not mutate state.** `state` is a
   plain dict snapshot handed to your node; assigning `state["x"] = ...` and
   returning `None` throws the write away. To pass a value from `before_model`
   to `after_model` you must (a) declare a channel for it on a custom state
   schema, and (b) `return {"x": ...}`.

The timing logger below does both. `TimingState` extends `AgentState` with one
`NotRequired` key, and `state_schema` tells the agent to merge that channel into
its own state.


In [6]:
# ============ A CUSTOM MIDDLEWARE ============
import time

from typing_extensions import NotRequired

from langchain.agents.middleware import AgentMiddleware, AgentState


class TimingState(AgentState):
    """AgentState plus one extra channel for the timer handoff."""

    model_call_started_at: NotRequired[float]


class TimingLoggerMiddleware(AgentMiddleware[TimingState]):
    """Logs latency around each model call in the agent loop."""

    state_schema = TimingState

    def before_model(self, state, runtime):
        print(">> calling model...")
        return {"model_call_started_at": time.time()}  # a state UPDATE

    def after_model(self, state, runtime):
        started = state.get("model_call_started_at")
        if started is not None:
            print(f"<< model responded in {time.time() - started:.2f}s")
        return None  # nothing to write back


# Note: for timing specifically, the `wrap_model_call` hook is simpler still --
# it brackets the call in a single method, so no state channel is needed at all.
# We use before/after here because it generalizes to hooks that must inspect
# state between steps.


In [7]:
agent_with_logging = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[TimingLoggerMiddleware()],
)

result = agent_with_logging.invoke({
    "messages": [{"role": "user", "content": "What's the status of account 1029?"}]
})
print("\nFinal answer:", result["messages"][-1].content)


>> calling model...
<< model responded in 1.35s
>> calling model...
<< model responded in 1.36s

Final answer: The status of account 1029 is active. The current balance is ₹42,500. Is there anything else you would like to know?


## 3. Stacking middleware

Middlewares run in the order you list them. A typical production stack
mirrors what you'd hand-roll as custom LangGraph nodes:


In [8]:
# ============ COMPOSING BUILT-IN + CUSTOM MIDDLEWARE ============
production_agent = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        ModelCallLimitMiddleware(run_limit=8),
        TimingLoggerMiddleware(),
    ],
)

# `create_agent` returns a compiled LangGraph, not a wrapper object - there is
# no `.middleware` attribute on it. Inspect the graph instead:
print(type(production_agent).__name__)
print("Nodes:", list(production_agent.get_graph().nodes))


CompiledStateGraph
Nodes: ['__start__', 'model', 'tools', 'PIIMiddleware[email].before_model', 'PIIMiddleware[email].after_model', 'ModelCallLimitMiddleware.before_model', 'ModelCallLimitMiddleware.after_model', 'TimingLoggerMiddleware.before_model', 'TimingLoggerMiddleware.after_model', '__end__']


## Recap

- Middleware replaces most of the reasons people used to drop into raw
  LangGraph just to add a cross-cutting concern (logging, PII redaction,
  call limits, summarization).
- You still write custom logic — but as a portable `AgentMiddleware`
  subclass, not a bespoke graph node tied to one project.

Next notebook: **03 — Human-in-the-loop middleware**, one specific,
pre-built middleware for pausing on risky tool calls.
